In [ ]:
# Load the autoreload extension
%load_ext autoreload

# Set autoreload mode
%autoreload 2

In [ ]:
import json

import fsspec
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

# CoralNet Annotation EDA

In [ ]:
df_annotations = pd.read_parquet(
    "s3://dev-datamermaid-sm-sources/etl-outputs/coralnet/20260526_807b611/coralnet_training_resized_20260526_807b611.parquet"
)

In [ ]:
df_annotations.head()

In [ ]:
print(
    f"We currently have {df_annotations.shape[0]} annotations within {df_annotations['image_id'].nunique()} unique images across {df_annotations['source_id'].nunique()} sources."
)

## Update CoralNet Label ID's to CoralNet Label Names

In [ ]:
s3_path = "s3://dev-datamermaid-sm-sources/coralnet-public-images/temporary/coralnet_id2name.json"

with fsspec.open(s3_path, "r") as f:
    coralnet_id2name = json.load(f)

In [ ]:
df_annotations["coralnet_name"] = df_annotations["coralnet_id"].map(
    lambda x: coralnet_id2name.get(str(x))
)
df_annotations["source_label_name"] = df_annotations["coralnet_name"]

In [ ]:
df_annotations["coralnet_id"].isna().sum(), df_annotations["source_label_name"].isna().sum()

In [ ]:
df_annotations[df_annotations["source_label_name"].isna()]["coralnet_id"].value_counts()

Interestingly the 2474 ID is mapped to None in the labelset mapping itself (rather than being missing in the mapping) - https://coralnet.ucsd.edu/label/2474/.    

## Explorations of the image sizes

In [ ]:
image_max_size = df_annotations.drop_duplicates(subset=["image_id"])[
    ["load_width", "load_height"]
].max(axis=1)

print(f"There are {(image_max_size < 2048).sum()} images with a maximum size less than 2048.")
print(
    f"There are {(image_max_size <= 512).sum()} images with a maximum size less than or equal to 512."
)

fig, ax = plt.subplots(figsize=(8, 6), nrows=2, layout="compressed")
image_max_size.plot.hist(bins=50, ax=ax[0])
ax[0].set_title("Distribution of Maximum Image Size")
image_max_size[image_max_size < 2048].plot.hist(bins=50, ax=ax[1])
ax[1].set_title("Distribution of Images with Maximum Size Less Than 2048")
plt.show()

In [ ]:
image_ratio = df_annotations.drop_duplicates(subset=["image_id"])[
    ["load_width", "load_height"]
].eval("load_width / load_height")

print(f"The minimum width to height ratio is {image_ratio.min()}.")
print(f"The maximum width to height ratio is {image_ratio.max()}.")

fig, ax = plt.subplots(figsize=(8, 3.5), layout="compressed")
image_ratio.plot.hist(bins=50, ax=ax)
ax.set_title("Distribution of Width to Height Ratios")
plt.show()

## Initial source subsetting

In [ ]:
df_source = df_annotations.groupby("source_id", as_index=False).agg(
    image_count=("image_id", "nunique"),
    annotations_count=("coralnet_id", "size"),
    num_labels=("coralnet_name", "nunique"),
)

In [ ]:
num_images_threshold = 100
num_annotations_threshold = 1000

source_mask = (df_source["image_count"] > num_images_threshold) & (
    df_source["annotations_count"] >= num_annotations_threshold
)

print(f"Sources with all updated (example) requirements (MERMAID): {source_mask.sum()}")

df_source.loc[source_mask, ["image_count", "annotations_count"]].sum()

In [ ]:
source_min_requirement_list = df_source[source_mask]["source_id"].to_numpy().tolist()
df_annotations_subset = df_annotations[
    df_annotations["source_id"].isin(source_min_requirement_list)
].reset_index(drop=True)

In [ ]:
df_annotations_subset = df_annotations_subset[
    df_annotations_subset["source_id"].isin(source_min_requirement_list)
].reset_index(drop=True)
print(
    f"After filtering based on the number of images and annotations, we currently have {df_annotations_subset.shape[0]} annotations within {df_annotations_subset['image_id'].nunique()} unique images across {df_annotations_subset['source_id'].nunique()} sources."
)

## Subset based on initial manual source assessment

In [ ]:
df_source_quality = pd.read_csv(
    "s3://dev-datamermaid-sm-sources/coralnet-public-images/temporary/coralnet_source_quality.csv"
)
source_quality_list = df_source_quality[df_source_quality["ToKeep"] == "Yes"]["Source ID"].tolist()

In [ ]:
df_annotations_subset = df_annotations_subset[
    df_annotations_subset["source_id"].isin(source_quality_list)
].reset_index(drop=True)
print(
    f"After filtering for high-quality sources, we currently have {df_annotations_subset.shape[0]} annotations within {df_annotations_subset['image_id'].nunique()} unique images across {df_annotations_subset['source_id'].nunique()} sources."
)

# Label Mappings Initialization

In [ ]:
df_labels_coralnet = df_annotations_subset["source_label_name"].value_counts().reset_index()

## MERMAID Mapping

In [ ]:
df_labelset100 = pd.read_csv(
    "s3://dev-datamermaid-sm-sources/coralnet-public-images/temporary/mapped_to_mermaid_attributes.csv"
)
MERMAID_labelset = set(df_labelset100["name"].values)
df_labelset100_top = df_labelset100[df_labelset100["top100"] == "1"]

In [ ]:
print(
    f"There are {len(MERMAID_labelset)} unique labels within the MERMAID label set, of which {len(df_labelset100_top)} are identified as high priority."
)

In [ ]:
df_labelset100_top.head()

In [ ]:
from mermaidseg.dataset_reconciliation import initialize_benthic_hierarchy
from mermaidseg.dataset_reconciliation.registry import roll_up_label

hierarchy_dict = initialize_benthic_hierarchy()
MERMAID_labelset_top100 = set(df_labelset100_top["name"].to_numpy())

mermaid_to_mermaid100_mapping = {}
for label in MERMAID_labelset:
    mermaid_to_mermaid100_mapping[label] = roll_up_label(
        label, hierarchy_dict, MERMAID_labelset_top100
    )

## CoralNet to MERMAID

In [ ]:
from mermaidseg.dataset_reconciliation import fetch_coralnet_to_mermaid

coralnet_to_mermaid_mapping = fetch_coralnet_to_mermaid()

In [ ]:
df_labels_coralnet["mermaid_label"] = df_labels_coralnet["source_label_name"].map(
    coralnet_to_mermaid_mapping
)

In [ ]:
df_labels_coralnet["mermaid_top_100"] = df_labels_coralnet["mermaid_label"].map(
    mermaid_to_mermaid100_mapping
)

## Source to Target Updated Mapping

In [ ]:
df_mapping = pd.read_csv("../../../../configs/class_to_concepts.csv")
df_mapping = df_mapping[
    [
        "source_label_class_name",
        "source_dataset_source",
        "target_label_class_name",
        "target_dataset_source",
    ]
].drop_duplicates()
df_mapping_coralnet = df_mapping[df_mapping["source_dataset_source"] == "coralnet"]

In [ ]:
coralnet_to_mermaid_mapping_stt = {}
coralnet_to_mermaid_mapping_stt_dataset = {}

for _i, row in df_mapping_coralnet.iterrows():
    target_label = row.target_label_class_name
    target_dataset = row.target_dataset_source
    while True:
        if target_dataset == "mermaid":
            coralnet_to_mermaid_mapping_stt[row.source_label_class_name] = target_label
            coralnet_to_mermaid_mapping_stt_dataset[row.source_label_class_name] = target_dataset
            break
        else:
            target_row = df_mapping[
                (df_mapping["source_label_class_name"] == target_label)
                * (df_mapping["source_dataset_source"] == target_dataset)
            ]
            if target_row.shape[0] == 0 or pd.isna(target_row.iloc[0]["target_dataset_source"]):
                coralnet_to_mermaid_mapping_stt[row.source_label_class_name] = target_label
                coralnet_to_mermaid_mapping_stt_dataset[row.source_label_class_name] = (
                    target_dataset
                )
                break
            else:
                target_label = target_row.iloc[0]["target_label_class_name"]
                target_dataset = target_row.iloc[0]["target_dataset_source"]

In [ ]:
print(
    "Out of {} labels, {} are mapped to the mermaid dataset".format(
        len(coralnet_to_mermaid_mapping_stt_dataset),
        len([v for v in coralnet_to_mermaid_mapping_stt_dataset.values() if v == "mermaid"]),
    )
)

In [ ]:
df_labels_coralnet["mermaid_stt"] = df_labels_coralnet["source_label_name"].apply(
    lambda x: coralnet_to_mermaid_mapping_stt.get(str(x).lower())
)

# CoralNet Label EDA

In [ ]:
df_labels_coralnet_eda = df_labels_coralnet.copy()
# df_labels_coralnet_eda = df_labels_coralnet_eda[df_labels_coralnet_eda["count"]>=500]

In [ ]:
for column in ["source_label_name", "mermaid_label", "mermaid_top_100", "mermaid_stt"]:
    df_labels_coralnet_eda[column] = df_labels_coralnet_eda[column].apply(lambda x: str(x).lower())

cols = ["source_label_name", "mermaid_label", "mermaid_top_100", "mermaid_stt"]

df_labels_coralnet_eda[cols] = df_labels_coralnet_eda[cols].replace(
    {
        "nan": None,
        "none": None,
        "None": None,
    }
)

In [ ]:
df_labels_coralnet_eda

In [ ]:
print(
    f"The original coralnet dataset contains {df_labels_coralnet['source_label_name'].nunique()} unique labels"
)
# print(f"Once we filter by only including labels with 500+ instances, the coralnet dataset contains {df_labels_coralnet_eda['source_label_name'].nunique()} unique labels")

## CoralNet API + Roll Up Label Mapping

In [ ]:
print(
    f"Using the coralnet->mermaid API label mapping, the coralnet dataset contains {df_labels_coralnet_eda['mermaid_label'].nunique()} unique labels"
)
print(
    f"After the roll up to only hand-selected labels, the coralnet dataset contains {df_labels_coralnet_eda['mermaid_top_100'].nunique()} unique labels"
)

In [ ]:
annotation_count_sum = df_labels_coralnet_eda["count"].sum()
annotation_count_sum_filtered = df_labels_coralnet_eda[
    df_labels_coralnet_eda["mermaid_top_100"].isna()
]["count"].sum()
print(
    f"In the mapping to the mermaid top 100 labels, we don't have a mapping for {df_labels_coralnet_eda['mermaid_top_100'].isna().sum()} out of {df_labels_coralnet_eda.shape[0]} labels, resulting in {annotation_count_sum_filtered} annotations remaining unmapped, which is {round(annotation_count_sum_filtered / annotation_count_sum * 100, 2)}% of the dataset."
)

In [ ]:
df_labels_coralnet_eda[df_labels_coralnet_eda["mermaid_top_100"].isna()].head(10)

## Jonathan's Mapping

In [ ]:
print(
    f"Using Jonathan's label mapping, the coralnet dataset contains {df_labels_coralnet_eda['mermaid_stt'].nunique()} unique labels"
)

In [ ]:
annotation_count_sum = df_labels_coralnet_eda["count"].sum()
annotation_count_sum_filtered = df_labels_coralnet_eda[
    df_labels_coralnet_eda["mermaid_stt"].isna()
]["count"].sum()
print(
    f"In the mapping to Jonathan's labels, we don't have mapping for {df_labels_coralnet_eda['mermaid_stt'].isna().sum()} out of {df_labels_coralnet_eda.shape[0]} labels, resulting in {annotation_count_sum_filtered} annotations remaining unmapped, which is {round(annotation_count_sum_filtered / annotation_count_sum * 100, 2)}% of the dataset."
)

In [ ]:
df_labels_coralnet_eda[df_labels_coralnet_eda["mermaid_stt"].isna()].head(10)

Differences between the two mappings

In [ ]:
label_mask = (
    df_labels_coralnet_eda["mermaid_label"].notna() * df_labels_coralnet_eda["mermaid_stt"].notna()
)
df_labels_coralnet_eda.loc[label_mask].loc[
    df_labels_coralnet_eda.loc[label_mask, "mermaid_label"]
    != df_labels_coralnet_eda.loc[label_mask, "mermaid_stt"]
]

## Result: Update Missing Mermaid Maps with Jonathan's Mapping
In the meantime we also add a few select labels to the MermaidTop100 Labelset

In [ ]:
manual_updates = {
    "hard substrate": "bare substrate",
    "": None,
    "hard coral alive": "hard coral",
    "seriatopora alive": "seriatopora",
    "transect line": "transect tools",
    "pocillopora alive": "pocillopora",
    "pocillopora dead": "pocillopora",
    "acropora alive": "acropora",
}

to_add = [
    "dark",
    "algae covered substrate",
    "other coral bleached",
    "transect tools",
    "other coral dead",
    "human-made structure",
    "background",
    "fish",
    "giant clam",
]
important = ["dark", "transect tools", "background", "fish"]

MERMAID_labelset_top100_upd = MERMAID_labelset_top100.union(to_add)
MERMAID_labelset_top100_upd = {label.lower() for label in MERMAID_labelset_top100_upd}
hierarchy_dict_lowercase = {
    k.lower() if k else None: v.lower() if v else None for k, v in hierarchy_dict.items()
}

In [ ]:
coralnet_to_mermaid_mapping_upd = {
    str(k).lower(): str(v).lower() for k, v in coralnet_to_mermaid_mapping.items() if v is not None
}
mermaid_to_mermaid100_mapping_upd = {
    str(k).lower(): str(v).lower()
    for k, v in mermaid_to_mermaid100_mapping.items()
    if v is not None
}

In [ ]:
coralnet_to_mermaid_mapping_final = {}
for source_label_name in df_labels_coralnet_eda["source_label_name"].unique():
    map_flag = False
    if source_label_name in coralnet_to_mermaid_mapping_upd:
        mermaid_label = coralnet_to_mermaid_mapping_upd[source_label_name]
        mermaid_label = manual_updates.get(mermaid_label, mermaid_label)
        mermaid_label = roll_up_label(
            mermaid_label, hierarchy_dict_lowercase, MERMAID_labelset_top100_upd
        )
        if mermaid_label is not None:
            coralnet_to_mermaid_mapping_final[source_label_name] = mermaid_label
            map_flag = True
    if source_label_name in coralnet_to_mermaid_mapping_stt and not map_flag:
        jonathan_label = coralnet_to_mermaid_mapping_stt[source_label_name]
        jonathan_label = manual_updates.get(jonathan_label, jonathan_label)
        coralnet_to_mermaid_mapping_final[source_label_name] = roll_up_label(
            jonathan_label, hierarchy_dict_lowercase, MERMAID_labelset_top100_upd
        )
        map_flag = True
    if not map_flag:
        coralnet_to_mermaid_mapping_final[source_label_name] = None
coralnet_to_mermaid_mapping_final = {
    k: v.capitalize() if v is not None else None
    for k, v in coralnet_to_mermaid_mapping_final.items()
}

In [ ]:
with open("../../../../configs/coralnet_to_mermaid_mapping_temporary.json", "w") as f:
    json.dump(coralnet_to_mermaid_mapping_final, f)

In [ ]:
df_labels_coralnet_eda["mermaid_final"] = df_labels_coralnet_eda["source_label_name"].apply(
    lambda x: coralnet_to_mermaid_mapping_final.get(str(x).lower())
)

In [ ]:
print(
    f"Using Jonathan's label mapping, the coralnet dataset contains {df_labels_coralnet_eda['mermaid_final'].nunique()} unique labels"
)

In [ ]:
annotation_count_sum = df_labels_coralnet_eda["count"].sum()
annotation_count_sum_filtered = df_labels_coralnet_eda[
    df_labels_coralnet_eda["mermaid_final"].isna()
]["count"].sum()
print(
    f"In the final mapping, we don't have mapping for {df_labels_coralnet_eda['mermaid_final'].isna().sum()} out of {df_labels_coralnet_eda.shape[0]} labels, resulting in {annotation_count_sum_filtered} annotations remaining unmapped, which is {round(annotation_count_sum_filtered / annotation_count_sum * 100, 2)}% of the dataset."
)

In [ ]:
df_labels_coralnet_eda[df_labels_coralnet_eda["mermaid_final"].isna()].head(10)

In [ ]:
count_sum_per_mermaid_final = (
    df_labels_coralnet_eda.groupby("mermaid_final", dropna=False)["count"]
    .sum()
    .sort_values(ascending=False)
    .reset_index(name="count_sum")
    .rename(columns={"mermaid_final": "mermaid_final_label"})
)

count_sum_per_mermaid_final

# Map Annotations using Finalized Label Mapping

In [ ]:
df_source_stats = df_annotations.groupby("source_id", as_index=False).agg(
    image_count=("image_id", "nunique"),
    annotations_count=("coralnet_id", "size"),
    num_labels=("coralnet_name", "nunique"),
)

In [ ]:
df_annotations_subset["mermaid_final"] = df_annotations_subset["source_label_name"].apply(
    lambda x: coralnet_to_mermaid_mapping_final.get(str(x).lower())
)

In [ ]:
df_annotations_subset.head()

In [ ]:
df_labels = df_annotations_subset.groupby("mermaid_final", as_index=False).agg(
    num_annotations=("mermaid_final", "size"),
    num_images=("image_id", "nunique"),
    num_sources=("source_id", "nunique"),
)

df_labels

## Final label subsetting

In [ ]:
num_images_threshold = 100
num_annotations_threshold = 500
num_sources_threshold = 3
df_labels_subset = df_labels[
    (df_labels["num_annotations"] >= num_annotations_threshold)
    * (df_labels["num_images"] >= num_images_threshold)
    * (df_labels["num_sources"] >= num_sources_threshold)
]
df_labels_subset

In [ ]:
df_labels_subset.describe()

In [ ]:
print([label.capitalize() for label in df_labels_subset["mermaid_final"].to_numpy()])

In [ ]:
df_source_labels_eligible = (
    df_annotations_subset.dropna(subset=["mermaid_final"])
    .groupby(["mermaid_final", "source_id"], as_index=False)
    .agg(
        num_annotations=("image_id", "size"),
        num_images=("image_id", "nunique"),
    )
    .sort_values(["mermaid_final", "num_annotations"], ascending=[True, False])
    .reset_index(drop=True)
)

df_source_labels_eligible.head()

In [ ]:
df_source_labels_eligible = df_source_labels_eligible[
    df_source_labels_eligible["mermaid_final"].isin(df_labels_subset["mermaid_final"])
]
df_source_labels_eligible

In [ ]:
df_source_labels_eligible[["mermaid_final", "source_id"]].nunique()

In [ ]:
print(
    "We are finally left with",
    df_source_labels_eligible[["num_annotations"]].sum().item(),
    "annotations and",
    df_source_labels_eligible[["mermaid_final"]].nunique().item(),
    "labels and",
    df_source_labels_eligible[["source_id"]].nunique().item(),
    "sources.",
)

In [ ]:
df_source_labels_eligible.groupby("source_id")[["num_annotations"]].sum().describe()

# Split Optimization

In [ ]:
import pandas as pd


def constrained_stratified_group_split_from_aggregated(
    df_agg,
    source_col="Source ID",
    label_col="Label ID",
    count_col="Num Annotations",  # or 'Num Images' depending on what you want to balance
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    n_iterations=5000,
    random_state=42,
    verbose=True,
):
    """
    Split pre-aggregated dataset with:
    - No source leakage across splits
    - Stratified labels based on annotation counts
    - Respects source-level grouping

    Parameters:
    -----------
    df_agg : pd.DataFrame
        Pre-aggregated dataframe with columns: source, label, count
    source_col : str
        Column identifying unique sources (e.g., 'Source ID')
    label_col : str
        Column with target labels (e.g., 'Label ID')
    count_col : str
        Column with counts to balance (e.g., 'Num Annotations')
    train_ratio, val_ratio, test_ratio : float
        Desired split proportions (should sum to 1)
    n_iterations : int
        Number of random assignments to try (more = better balance)
    random_state : int
        Random seed for reproducibility
    verbose : bool
        Print progress and statistics
    """
    np.random.seed(random_state)

    # Validate inputs
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-10, "Ratios must sum to 1"

    # 1. Get unique sources and labels
    sources = df_agg[source_col].unique()
    unique_labels = df_agg[label_col].unique()
    n_sources = len(sources)
    n_labels = len(unique_labels)

    if verbose:
        print(f"Found {n_sources} unique sources")
        print(f"Found {n_labels} unique labels")
        print(f"Total {count_col}: {df_agg[count_col].sum():,}")

    # 2. Create source-label count matrix
    # Initialize matrix: sources × labels
    label_matrix = np.zeros((n_sources, n_labels))
    source_to_idx = {source: i for i, source in enumerate(sources)}
    # Reorder labels by rarity (rare labels first) to help the optimizer
    label_source_counts = (
        df_agg.groupby(label_col)[source_col].nunique().reindex(unique_labels, fill_value=0)
    )
    unique_labels = label_source_counts.sort_values().index.to_numpy()
    label_to_idx = {label: j for j, label in enumerate(unique_labels)}

    # Hard feasibility check:
    # with group-based splitting, each label must exist in >= 3 different sources
    # to be present in train/val/test simultaneously.
    impossible_labels = label_source_counts[label_source_counts < 3]
    if not impossible_labels.empty:
        preview = ", ".join(
            f"{lbl} ({cnt} sources)" for lbl, cnt in impossible_labels.head(10).items()
        )
        raise ValueError(
            "Cannot guarantee every label in every split. "
            f"These labels appear in fewer than 3 sources: {preview}"
            + (" ..." if len(impossible_labels) > 10 else "")
        )

    for _, row in df_agg.iterrows():
        i = source_to_idx[row[source_col]]
        j = label_to_idx[row[label_col]]
        label_matrix[i, j] = row[count_col]

    # 3. Calculate source sizes (total annotations per source)
    source_sizes = label_matrix.sum(axis=1)

    # 4. Calculate global label distribution (target proportions)
    total_annotations = label_matrix.sum()
    global_label_dist = label_matrix.sum(axis=0) / total_annotations

    if verbose:
        print("\nGlobal label distribution (top 5):")
        for j in np.argsort(-global_label_dist)[:5]:
            print(f"  Label {unique_labels[j]}: {global_label_dist[j]:.2%}")

    # 5. Iterative random assignment with scoring
    best_score = np.inf
    best_assignment = None

    target_sizes = np.array([train_ratio, val_ratio, test_ratio]) * total_annotations

    if verbose:
        print("\nTarget split sizes:")
        print(f"  Train: {target_sizes[0]:,.0f} annotations ({train_ratio:.1%})")
        print(f"  Val:   {target_sizes[1]:,.0f} annotations ({val_ratio:.1%})")
        print(f"  Test:  {target_sizes[2]:,.0f} annotations ({test_ratio:.1%})")
        print(f"\nRunning {n_iterations} iterations...")

    for iteration in range(n_iterations):
        if verbose and iteration % 1000 == 0:
            print(f"  Iteration {iteration}/{n_iterations}")

        # Randomly shuffle sources for greedy assignment
        shuffled_idx = np.random.permutation(n_sources)
        assignments = np.zeros(n_sources, dtype=int)
        split_counts = np.zeros((3, n_labels))
        split_sizes = np.zeros(3)

        # Greedy assignment
        for idx in shuffled_idx:
            source_label_counts = label_matrix[idx]
            source_size = source_sizes[idx]

            # Calculate imbalance if added to each split
            scores = []
            for split_id in range(3):
                current_size = split_sizes[split_id]

                # Penalize if split would exceed target by >20%
                if current_size + source_size > target_sizes[split_id] * 1.2:
                    scores.append(np.inf)
                else:
                    new_counts = split_counts[split_id] + source_label_counts
                    new_size = current_size + source_size

                    # Expected counts for this split size
                    expected = new_size * global_label_dist

                    # Chi-square distance from expected proportions
                    # Add small epsilon to avoid division by zero
                    score = np.sum((new_counts - expected) ** 2 / (expected + 1e-10))

                    # Add penalty for deviating from target size
                    size_penalty = (
                        abs(new_size - target_sizes[split_id]) / target_sizes[split_id] * 10
                    )
                    score += size_penalty

                    scores.append(score)

            # Assign to split with minimum score
            best_split = np.argmin(scores)
            assignments[idx] = best_split
            split_counts[best_split] += source_label_counts
            split_sizes[best_split] += source_size

        # Calculate overall score (lower is better)
        overall_score = 0
        for split_id in range(3):
            expected = split_sizes[split_id] * global_label_dist
            overall_score += np.sum((split_counts[split_id] - expected) ** 2 / (expected + 1e-10))

        if overall_score < best_score:
            best_score = overall_score
            best_assignment = assignments.copy()

            if verbose:
                print(f"    New best score: {best_score:.2f}")

    if verbose:
        print(
            f"\nOptimization complete after {n_iterations} iterations! Best score: {best_score:.2f}"
        )

    # 6. Create split mapping
    split_map = {0: "train", 1: "val", 2: "test"}
    source_to_split = {source: split_map[best_assignment[i]] for i, source in enumerate(sources)}

    # 7. Add split column to dataframe
    df_result = df_agg.copy()
    df_result["split"] = df_result[source_col].map(source_to_split)

    # 8. Print summary statistics
    if verbose:
        print("\n" + "=" * 50)
        print("SPLIT SUMMARY")
        print("=" * 50)

        for split_name in ["train", "val", "test"]:
            split_df = df_result[df_result["split"] == split_name]
            split_annotations = split_df[count_col].sum()
            split_sources = split_df[source_col].nunique()

            print(f"\n{split_name.upper()}:")
            print(
                f"  Annotations: {split_annotations:,} ({split_annotations / total_annotations:.2%})"
            )
            print(f"  Sources: {split_sources} ({split_sources / n_sources:.2%})")

            # Label distribution check
            label_dist = split_df.groupby(label_col)[count_col].sum()
            label_dist_pct = label_dist / label_dist.sum()
            print(
                f"  Top 3 labels: {', '.join([f'{lbl}: {pct:.2%}' for lbl, pct in label_dist_pct.nlargest(3).items()])}"
            )

    return df_result

In [ ]:
def validate_aggregated_splits(
    df_with_splits, source_col="source_id", label_col="mermaid_final", count_col="num_annotations"
):
    """
    Comprehensive validation of the split results
    """
    print("=" * 60)
    print("VALIDATION RESULTS")
    print("=" * 60)

    # 1. Source disjointness (CRITICAL)
    source_split_counts = df_with_splits.groupby(source_col)["split"].nunique()
    sources_in_multiple_splits = (source_split_counts > 1).sum()

    if sources_in_multiple_splits == 0:
        print("✓ PASS: Each source appears in exactly one split")
    else:
        print(f"✗ FAIL: {sources_in_multiple_splits} sources appear in multiple splits!")
        print("  Problematic sources:", source_split_counts[source_split_counts > 1].index.tolist())

    # 2. Label distribution similarity
    print("\nLabel Distribution by Split:")
    label_totals = df_with_splits.groupby("split")[count_col].sum()
    global_label_dist = df_with_splits.groupby(label_col)[count_col].sum() / label_totals.sum()

    comparison_df = pd.DataFrame()
    for split in ["train", "val", "test"]:
        split_data = df_with_splits[df_with_splits["split"] == split]
        split_dist = split_data.groupby(label_col)[count_col].sum() / split_data[count_col].sum()
        comparison_df[split] = split_dist

    comparison_df["global"] = global_label_dist
    comparison_df["max_deviation"] = (
        comparison_df[["train", "val", "test"]]
        .sub(comparison_df["global"], axis=0)
        .abs()
        .max(axis=1)
    )

    print(f"  Average label distribution deviation: {comparison_df['max_deviation'].mean():.4f}")
    print(f"  Max label distribution deviation: {comparison_df['max_deviation'].max():.4f}")

    # 3. Split sizes
    print("\nSplit Sizes:")
    for split in ["train", "val", "test"]:
        split_total = df_with_splits[df_with_splits["split"] == split][count_col].sum()
        pct = split_total / label_totals.sum()
        print(f"  {split}: {split_total:,} annotations ({pct:.2%})")

    # 4. Coverage check
    print(f"\nTotal unique labels: {df_with_splits[label_col].nunique()}")
    for split in ["train", "val", "test"]:
        split_labels = df_with_splits[df_with_splits["split"] == split][label_col].nunique()
        coverage = split_labels / df_with_splits[label_col].nunique()
        print(
            f"  {split} label coverage: {split_labels}/{df_with_splits[label_col].nunique()} ({coverage:.2%})"
        )
    comparison_df = comparison_df[["train", "val", "test"]].div(comparison_df["global"], axis=0)
    comparison_df["std"] = comparison_df[["train", "val", "test"]].std(axis=1)
    return comparison_df

In [ ]:
df_with_splits = constrained_stratified_group_split_from_aggregated(
    df_agg=df_source_labels_eligible,
    source_col="source_id",
    label_col="mermaid_final",
    count_col="num_annotations",
    train_ratio=0.6,
    val_ratio=0.2,
    test_ratio=0.2,
    n_iterations=15000,
    random_state=44,
)

In [ ]:
## We swap the val and test temporarily just to get a better split
df_with_splits["split"] = df_with_splits["split"].replace({"val": "temp", "test": "val"})
df_with_splits["split"] = df_with_splits["split"].replace({"temp": "test"})

In [ ]:
df_source_splits = df_with_splits.groupby("source_id", as_index=False).agg(
    num_annotations=("num_annotations", "sum"),
    split=("split", "first"),
    num_unique_labels=("mermaid_final", "nunique"),
)

df_source_splits = df_source_splits.merge(
    df_source[["source_id", "image_count"]], on="source_id", how="left"
)

In [ ]:
df_source_splits.groupby("split", as_index=False)[["num_annotations", "image_count"]].sum()

In [ ]:
df_source_splits.groupby("split", as_index=False)[["num_annotations", "image_count"]].sum()

In [ ]:
print(df_source_splits[df_source_splits["split"] == "train"]["source_id"].tolist())

In [ ]:
print(df_source_splits[df_source_splits["split"] == "val"]["source_id"].tolist())

In [ ]:
print(df_source_splits[df_source_splits["split"] == "test"]["source_id"].tolist())

In [ ]:
# Run validation
comparison_df = validate_aggregated_splits(df_with_splits)

In [ ]:
comparison_df.sort_values("std", ascending=False)

# Final EDA on CoralNet Subset Currently Used

In [ ]:
whitelist_sources_train = [
    1645,
    1783,
    1968,
    2055,
    3058,
    3064,
    3363,
    3401,
    3411,
    3425,
    3460,
    3478,
    3681,
    4342,
    4559,
    4721,
    4882,
    5155,
    5181,
    5847,
    6053,
    6537,
    6687,
    6698,
    6716,
    6721,
    6722,
    6733,
    6839,
    6855,
    6868,
    6920,
    6924,
    6929,
    6931,
    6932,
]
whitelist_sources_val = [
    1776,
    2297,
    2759,
    2978,
    3573,
    4353,
    4596,
    4710,
    5086,
    5444,
    6521,
    6699,
    6700,
    6717,
    6834,
    6838,
    6923,
    6930,
    6968,
    7116,
]
whitelist_sources_test = [4430, 4451, 4514, 5221, 6695, 6835, 6921, 6922, 6928, 7054]
whitelist_sources_all = set(
    whitelist_sources_train + whitelist_sources_val + whitelist_sources_test
)

In [ ]:
label_subset = [
    "Acanthastrea",
    "Acropora",
    "Agaricia agaricites",
    "Algae covered substrate",
    "Alveopora",
    "Astrea",
    "Astreopora",
    "Background",
    "Bare substrate",
    "Coscinaraea",
    "Crustose coralline algae",
    "Cyanobacteria",
    "Cyphastrea",
    "Dark",
    "Dictyota",
    "Diploastrea",
    "Dipsastraea",
    "Echinophyllia",
    "Echinopora",
    "Euphyllia",
    "Favia",
    "Favites",
    "Fungia",
    "Galaxea",
    "Gardineroseris",
    "Goniastrea",
    "Goniopora",
    "Gorgonian",
    "Halimeda",
    "Hard coral",
    "Heliopora",
    "Human-made structure",
    "Hydnophora",
    "Isopora",
    "Leptastrea",
    "Leptoria",
    "Leptoseris",
    "Lobophora",
    "Lobophyllia",
    "Macroalgae",
    "Madracis",
    "Merulina",
    "Millepora",
    "Montastraea",
    "Montastraea cavernosa",
    "Montipora",
    "Mycedium",
    "Orbicella annularis",
    "Orbicella faveolata",
    "Pachyseris",
    "Padina",
    "Pavona",
    "Pectinia",
    "Platygyra",
    "Pocillopora",
    "Porites",
    "Porites astreoides",
    "Psammocora",
    "Rock",
    "Rubble",
    "Sand",
    "Sargassum",
    "Sea urchin",
    "Seagrass",
    "Seriatopora",
    "Siderastrea siderea",
    "Soft coral",
    "Sponge",
    "Stylophora",
    "Symphyllia",
    "Transect tools",
    "Tridacna giant clam",
    "Turbinaria-algae",
    "Turbinaria-coral",
    "Turf algae",
    "Zoanthid",
]
label_subset = [label.lower() for label in label_subset]

In [ ]:
df_annotations_final = df_annotations
df_annotations_final = df_annotations_final[
    df_annotations_final["source_id"].isin(whitelist_sources_all)
]
df_annotations_final["target_label_name"] = df_annotations_final["source_label_name"].apply(
    lambda x: coralnet_to_mermaid_mapping_final.get(str(x).lower())
)
df_annotations_final = df_annotations_final[
    df_annotations_final["target_label_name"].isin(label_subset)
]
df_annotations_final = df_annotations_final.reset_index(drop=True)

In [ ]:
df_annotations_final.head()

In [ ]:
print(
    f"After the final filtering, we currently have {df_annotations_final.shape[0]} annotations within {df_annotations_final['image_id'].nunique()} unique images across {df_annotations_final['source_id'].nunique()} sources."
)

# Source EDA

In [ ]:
df_source = (
    df_annotations_final.groupby("source_id")
    .agg(
        image_count=("image_id", "nunique"),
        annotations_count=("image_id", "size"),
        num_labels_coralnet=("source_label_name", "nunique"),
        num_labels_mermaid=("target_label_name", "nunique"),
    )
    .reset_index()
)

df_source.head()

In [ ]:
df_source.shape

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), nrows=2, ncols=2)
df_source["image_count"].hist(bins=30, edgecolor="white", color="#1C124A", ax=ax[0, 0])
ax[0, 0].set_xlabel("image_count")

np.log10(df_source["image_count"]).hist(bins=30, edgecolor="white", color="#1C124A", ax=ax[0, 1])
ax[0, 1].set_xlabel("log10(image_count)")

df_source["annotations_count"].hist(bins=30, edgecolor="white", color="#1C124A", ax=ax[1, 0])
ax[1, 0].set_xlabel("annotations_count")

np.log10(df_source["annotations_count"]).hist(
    bins=30, edgecolor="white", color="#1C124A", ax=ax[1, 1]
)
ax[1, 1].set_xlabel("log10(annotations_count)")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), nrows=2)
df_source["num_labels_coralnet"].hist(bins=30, edgecolor="white", color="#1C124A", ax=ax[0])
ax[0].set_xlabel("num_labels_coralnet")

df_source["num_labels_mermaid"].hist(bins=30, edgecolor="white", color="#1C124A", ax=ax[1])
ax[1].set_xlabel("num_labels_mermaid")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5), ncols=2)
ax[0].scatter(df_source["image_count"], df_source["annotations_count"], alpha=0.6, color="#1C124A")

ax[0].set_xlabel("Number of images")
ax[0].set_ylabel("Number of annotations")
ax[0].set_title("Images vs Annotations per Source")

ax[1].scatter(
    np.log10(df_source["image_count"]),
    np.log10(df_source["annotations_count"]),
    alpha=0.6,
    color="#1C124A",
)

ax[1].set_xlabel("logNumber of images")
ax[1].set_ylabel("logNumber of annotations")
ax[1].set_title("Images vs Annotations per Source")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5), ncols=2)
ax[0].scatter(df_source["image_count"], df_source["num_labels_mermaid"], alpha=0.6, color="#1C124A")

ax[0].set_xlabel("Number of images")
ax[0].set_ylabel("Number of labels")
ax[0].set_title("Images vs Labels per Source")

ax[1].scatter(
    np.log10(df_source["image_count"]), df_source["num_labels_mermaid"], alpha=0.6, color="#1C124A"
)

ax[1].set_xlabel("logNumber of images")
ax[1].set_ylabel("Number of labels")
ax[1].set_title("Images vs Labels per Source")
plt.show()

# Label EDA

In [ ]:
df_labels = (
    df_annotations_final.groupby(["target_label_name"], dropna=False)
    .agg(
        num_annotations=("target_label_name", "size"),
        num_images=("image_id", "nunique"),
        num_sources=("source_id", "nunique"),
    )
    .reset_index()
)

df_labels.head()

In [ ]:
df_labels.shape

In [ ]:
fig, ax = plt.subplots(figsize=(7, 9), nrows=3, layout="compressed")
ax[0].hist(np.log10(df_labels["num_annotations"]), bins=50, edgecolor="white", color="#1C124A")
ax[0].set_xlabel("log(Num Annotations)")
ax[0].set_ylabel("Count")
ax[0].set_title("Histogram of Num Annotations per Label ID")

ax[1].hist(np.log10(df_labels["num_images"]), bins=50, edgecolor="white", color="#1C124A")
ax[1].set_xlabel("log(Num Images)")
ax[1].set_ylabel("Count")
ax[1].set_title("Histogram of Num Images per Label ID")

ax[2].hist(np.log10(df_labels["num_sources"]), bins=50, edgecolor="white", color="#1C124A")
ax[2].set_xlabel("log(Num Sources)")
ax[2].set_ylabel("Count")
ax[2].set_title("Histogram of Num Sources per Label ID")

plt.show()